# Tie groups and the section 7.4 case study

Section 2.3.3 defines one tie-group object. It is **not transitive**, so it is
not a partition and cannot be used as one. `docs/spec_addenda.md#g1` provides
three objects instead.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

## The dyadic ladder

`s_i = i * 2^-20`. Every value and every difference is exactly representable, so
this demonstration has no floating-point content of its own.

In [ ]:
from tfidf_stability.ranking.tie_groups import (
    chain_inflation_ratio,
    tie_ball_interval,
    tie_chains,
    tie_cliques,
)

n = 8
tau = 2.0 ** -20
ladder = sorted((i * tau for i in range(n)), reverse=True)

print("ball(j) for each j -- note these are NOT nested and NOT transitive:")
for j in range(n):
    lo, hi = tie_ball_interval(ladder, j, tau)
    print(f"  ball({j}) = [{lo}, {hi}]")

chains = tie_chains(ladder, tau)
cliques = tie_cliques(ladder, tau)
print(f"\nchains  {len(chains)}: {chains}")
print(f"cliques {len(cliques)}: {cliques}")

largest_chain = max(b - a for a, b in chains)  # half-open intervals
largest_clique = max(b - a for a, b in cliques)
print(f"\nrho = {largest_chain} / {largest_clique} = {largest_chain / largest_clique}")
print("one chain swallows the ladder; cliques see only adjacent pairs")

### The discontinuity in the diagnostic itself

One ulp below tau, the ladder shatters into singletons and rho drops to 1.

In [ ]:
import math

tighter = math.nextafter(tau, 0.0)
chains2 = tie_chains(ladder, tighter)
cliques2 = tie_cliques(ladder, tighter)
print(f"tau        -> {len(chains)} chains, rho = {largest_chain / largest_clique}")
print(f"tau - 1ulp -> {len(chains2)} chains, rho = "
      f"{max(b - a for a, b in chains2) / max(b - a for a, b in cliques2)}")

## Section 7.4: near-ties are identified, not constructed

A fine near-tie cannot be manufactured by editing text. Section 2.2's
`tf = count/L` makes a one-token edit a `1/(L+1)` **relative** perturbation, so a
separation of 1e-9 would need a billion-token document -- see
`docs/spec_addenda.md#g22`.

So the case study searches for the closest pair the corpus actually contains.

In [ ]:
from tfidf_stability.datasets.loaders import load_dataset
from tfidf_stability.datasets.synthetic import find_near_ties
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline
from tfidf_stability.similarity.cosine import cosine_against_corpus
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

data = load_dataset("synthetic_tiny")
pipeline = PreprocessingPipeline()
features = [pipeline.preprocess(str(r["text"])) for r in data.records]
model = TfidfVectoriser().fit(features, data.doc_ids)
documents = [model.document(i) for i in range(model.n_documents)]

scores = sorted(cosine_against_corpus(
    TfidfVectoriser.transform_query(list(features[0])[:6], model),
    documents, model.norms), reverse=True)

print("tightest gaps overall (exact ties included):")
for tie in find_near_ties(scores, limit=3, strictly_positive=False):
    print(f"  rank {tie.rank:3}  gap {tie.gap:.6e}  exact={tie.is_exact}")

print("\nclosest strictly-positive gaps:")
for tie in find_near_ties(scores, limit=3, strictly_positive=True):
    print(f"  rank {tie.rank:3}  gap {tie.gap:.6e}")